# 01 — Dari Model ke API: FastAPI dan Demo Gradio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/12-deployment/01_model_ke_api_fastapi.ipynb)

Model yang hanya hidup di notebook belum berguna bagi orang lain. **Deployment** berarti menyajikan model sebagai **layanan** yang bisa dipanggil aplikasi lain (web, mobile, sistem internal) kapan saja.

```
aplikasi / pengguna ──HTTP (JSON)──▶  API (FastAPI)  ──▶  model  ──▶  {"label": "positif", "skor": ...}
```

Studi kasus modul ini: **model sentimen bahasa Indonesia** kecil (embedding + MLP, ±140 ribu parameter) yang dilatih pada NusaX-Senti. Modelnya sengaja sederhana agar training cukup ±20 detik di CPU, sehingga fokus kita ada pada **deployment**. Untuk akurasi lebih tinggi, ganti dengan IndoBERT hasil fine-tuning di modul 06; pola deployment-nya sama.

Isi notebook:
1. Training dan **artefak** model.
2. API dengan **FastAPI**: validasi input, health check, batch, API key.
3. **Pengujian** API dengan `TestClient`.
4. Menjalankan server sungguhan dan mengukur **latensi**.
5. Demo UI dengan **Gradio**.

> Tidak butuh GPU dan tidak butuh API key.

## 0. Instalasi dan File Aplikasi

In [ ]:
%pip install -q fastapi uvicorn httpx gradio onnx onnxscript onnxruntime pandas

Kode aplikasi ada sebagai file biasa di folder [`app/`](app/) (bukan di dalam notebook), karena begitulah aplikasi sungguhan disusun: notebook untuk eksplorasi, file `.py` untuk kode yang di-deploy. Jika notebook dijalankan di Colab, file-file itu diunduh otomatis dari GitHub.

In [ ]:
import os
import sys
import urllib.request
from pathlib import Path

QUICK_RUN = os.environ.get("QUICK_RUN") == "1"
REPO_RAW = "https://raw.githubusercontent.com/riki-profile/ai-engineer-journey/main/12-deployment/app"
FILE_APP = ["model.py", "latih.py", "main.py", "ui_gradio.py", "requirements.txt", "Dockerfile", ".dockerignore",
            "tests/test_api.py"]
APP = Path("app")
for f in FILE_APP:
    if not (APP / f).exists():
        (APP / f).parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{REPO_RAW}/{f}", APP / f)

sys.path.insert(0, str(APP.resolve()))                         # agar `import model`, `import main` bisa dipakai
ARTEFAK = (APP / "artefak").resolve()
os.environ["FOLDER_MODEL"] = str(ARTEFAK)                      # konfigurasi API lewat environment variable
print(f"Folder aplikasi: {APP.resolve()} | QUICK_RUN: {QUICK_RUN}")

## 1. Training dan Artefak

[`app/latih.py`](app/latih.py) melatih model dan menyimpan **artefak**, yaitu semua yang dibutuhkan untuk memprediksi tanpa kode training:

| File | Isi | Mengapa penting |
|---|---|---|
| `model.pt` | Bobot PyTorch | Untuk eksperimen dan melanjutkan training |
| `model.onnx` | Model dalam format ONNX | Untuk serving tanpa PyTorch (notebook 02) |
| `vocab.json` | Kamus token → ID | Preprocessing saat serving **harus identik** dengan saat training |
| `config.json` | Versi, hyperparameter, metrik | Agar tahu model mana yang sedang berjalan dan seberapa bagus |

Kesalahan klasik di produksi adalah *training–serving skew*: preprocessing saat serving sedikit berbeda dari saat training (misalnya lupa huruf kecil), dan akurasi turun diam-diam. Solusinya: kode preprocessing yang **sama** ([`app/model.py`](app/model.py)) dipakai di kedua tempat, dan vocab disimpan bersama model.

In [ ]:
from latih import latih

config = latih(ARTEFAK, epoch=5 if QUICK_RUN else 40)
print(config)
for f in sorted(ARTEFAK.iterdir()):
    print(f"  {f.name:12} {f.stat().st_size / 1e3:8.1f} KB")

`Prediktor` membungkus semua langkah inference: teks → token → ID → model → probabilitas → label. Kelas inilah yang dipakai API, UI, dan tes.

In [ ]:
from model import Prediktor

prediktor = Prediktor(ARTEFAK, backend="torch")
contoh = ["Pengirimannya cepat, barangnya bagus!", "Pelayanannya lambat, kecewa.", "Saya membeli ini kemarin."]
for teks, hasil in zip(contoh, prediktor.prediksi(contoh)):
    print(f"{hasil['label']:8} {hasil['skor']}  ← {teks}")

## 2. API dengan FastAPI

**FastAPI** adalah framework web Python yang populer untuk API model ML: cepat, validasi input otomatis lewat **Pydantic**, dan dokumentasi interaktif (`/docs`) dibuat otomatis. Berikut isi [`app/main.py`](app/main.py):

In [ ]:
print((APP / "main.py").read_text())

Hal-hal penting di kode tersebut:
- **Model dimuat sekali** saat server mulai (`lifespan`), bukan di setiap permintaan. Memuat model bisa butuh detik hingga menit.
- **Validasi input** dengan Pydantic: teks wajib ada, 1–2000 karakter, batch maksimal 32. Input tidak valid ditolak dengan status **422** sebelum menyentuh model. Ini melindungi server dari input aneh atau terlalu besar.
- **Skema output** (`response_model`) membuat kontrak API jelas bagi pemakainya.
- Endpoint prediksi berupa fungsi `def` biasa (bukan `async def`): FastAPI menjalankannya di *thread pool*, sehingga komputasi CPU tidak memblokir permintaan lain.
- **`/health`** untuk pemeriksaan kesehatan oleh load balancer/Docker/Kubernetes.
- **API key** lewat header `X-API-Key`, nilainya dari environment variable `API_KEY` (bukan di kode), dan dibandingkan dengan `secrets.compare_digest`.
- **Middleware** mencatat waktu proses setiap permintaan ke log dan header `X-Waktu-Proses-ms`, dasar untuk monitoring (modul 13).

## 3. Menguji API dengan TestClient

`TestClient` memanggil API **tanpa** menjalankan server sungguhan: cepat dan cocok untuk tes otomatis. Repo ini punya tes lengkap di [`app/tests/test_api.py`](app/tests/test_api.py) yang dijalankan CI di setiap pull request.

In [ ]:
import logging

from fastapi.testclient import TestClient

import main

for nama_logger in ["httpx", "httpx2"]:                    # log klien HTTP tidak perlu ditampilkan
    logging.getLogger(nama_logger).setLevel(logging.WARNING)

with TestClient(main.app) as klien:          # `with` menjalankan lifespan (memuat model)
    print("GET /health             →", klien.get("/health").json())
    print("POST /prediksi          →", klien.post("/prediksi", json={"teks": "Mantap, recommended!"}).json())
    r = klien.post("/prediksi/batch", json={"teks": ["enak sekali", "pelayanannya buruk"]})
    print("POST /prediksi/batch    →", [h["label"] for h in r.json()])

    r = klien.post("/prediksi", json={"teks": ""})
    print(f"\nInput kosong → status {r.status_code}: {r.json()['detail'][0]['msg']}")
    r = klien.post("/prediksi/batch", json={"teks": ["x"] * 100})
    print(f"Batch 100 teks → status {r.status_code}: {r.json()['detail'][0]['msg']}")

API key: saat `API_KEY` diatur, permintaan tanpa header yang benar ditolak dengan **401**. Di sini kita mengubahnya sementara di kode hanya untuk demonstrasi; di produksi nilainya berasal dari environment variable (notebook 03).

In [ ]:
import secrets

main.API_KEY = secrets.token_urlsafe(16)      # kunci acak sementara untuk demo
with TestClient(main.app) as klien:
    data = {"teks": "bagus"}
    print("tanpa key :", klien.post("/prediksi", json=data).status_code)
    print("key salah :", klien.post("/prediksi", json=data, headers={"X-API-Key": "salah"}).status_code)
    print("key benar :", klien.post("/prediksi", json=data, headers={"X-API-Key": main.API_KEY}).status_code)
main.API_KEY = None

FastAPI juga membuat spesifikasi **OpenAPI** otomatis. Saat server berjalan, dokumentasi interaktifnya ada di `http://<server>/docs`, lengkap dengan tombol untuk mencoba setiap endpoint.

In [ ]:
skema = main.app.openapi()
for path, metode in skema["paths"].items():
    for m, info in metode.items():
        print(f"{m.upper():5} {path:18} {info.get('summary', '')}")

## 4. Server Sungguhan dan Latensi

Sekarang jalankan server **uvicorn** sungguhan (di thread latar belakang notebook) dan panggil lewat HTTP dengan `httpx`, persis seperti aplikasi lain memanggilnya. Kita ukur:
- **Latensi** per permintaan: median (p50) dan p95 (95% permintaan lebih cepat dari angka ini). Di produksi, p95/p99 sering lebih penting daripada rata-rata karena menentukan pengalaman pengguna yang "sial".
- **Throughput**: berapa teks per detik, dengan permintaan satu per satu vs batch vs paralel.

In [ ]:
import threading
import time
from concurrent.futures import ThreadPoolExecutor

import httpx
import numpy as np
import uvicorn

logging.getLogger("api-sentimen").setLevel(logging.WARNING)   # matikan log per permintaan selama uji beban
PORT = 8123
server = uvicorn.Server(uvicorn.Config(main.app, host="127.0.0.1", port=PORT, log_level="warning"))
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
while not server.started:
    time.sleep(0.1)
print(f"Server berjalan di http://127.0.0.1:{PORT}")

teks_uji = ["Barangnya bagus dan pengiriman cepat sekali, terima kasih!"] * (50 if QUICK_RUN else 200)
http = httpx.Client(base_url=f"http://127.0.0.1:{PORT}", trust_env=False)   # trust_env=False: jangan lewat proxy

latensi = []
mulai = time.perf_counter()
for t in teks_uji:
    t0 = time.perf_counter()
    http.post("/prediksi", json={"teks": t}).raise_for_status()
    latensi.append((time.perf_counter() - t0) * 1000)
satu_per_satu = len(teks_uji) / (time.perf_counter() - mulai)

mulai = time.perf_counter()
for i in range(0, len(teks_uji), 32):
    http.post("/prediksi/batch", json={"teks": teks_uji[i:i + 32]}).raise_for_status()
batch = len(teks_uji) / (time.perf_counter() - mulai)

mulai = time.perf_counter()
with ThreadPoolExecutor(max_workers=8) as pool:          # 8 "pengguna" bersamaan, berbagi satu koneksi pool
    list(pool.map(lambda t: http.post("/prediksi", json={"teks": t}).raise_for_status(), teks_uji))
paralel = len(teks_uji) / (time.perf_counter() - mulai)

print(f"Latensi per permintaan: p50 {np.percentile(latensi, 50):.1f} ms | p95 {np.percentile(latensi, 95):.1f} ms")
print(f"Throughput: satu per satu {satu_per_satu:,.0f} teks/detik | batch 32 {batch:,.0f} | 8 klien paralel {paralel:,.0f}")

Cara membaca hasilnya:
- **Batch** biasanya jauh lebih cepat per teks: overhead HTTP, parsing JSON, dan pemanggilan model dibayar sekali untuk banyak teks.
- **Permintaan paralel** sering *tidak* menaikkan throughput di sini. Server kita hanya satu proses Python, dan pekerjaan per permintaan sangat singkat serta dibatasi GIL (*Global Interpreter Lock*: hanya satu thread Python berjalan pada satu waktu). Untuk memanfaatkan beberapa core CPU, jalankan beberapa proses (`uvicorn main:app --workers 4`) atau beberapa replika container, dan manfaatkan batching.
- Waktu model sendiri hanya beberapa milidetik (lihat header `X-Waktu-Proses-ms`); sebagian besar latensi di sini adalah overhead jaringan dan framework. Untuk model besar keadaannya terbalik, sehingga optimasi model menjadi penting (notebook 02).

In [ ]:
r = http.post("/prediksi", json={"teks": "coba"})
print("Waktu proses di server:", r.headers["X-Waktu-Proses-ms"], "ms")

http.close()
server.should_exit = True           # hentikan server
thread.join(timeout=5)
logging.getLogger("api-sentimen").setLevel(logging.INFO)
print("Server dihentikan")

## 5. Demo UI dengan Gradio

API cocok untuk aplikasi lain. Untuk **manusia** (demo ke atasan, pengguna non-teknis, portofolio), kita butuh antarmuka. **Gradio** membuat UI web dari fungsi Python hanya dengan beberapa baris ([`app/ui_gradio.py`](app/ui_gradio.py)). Alternatif populer lainnya adalah **Streamlit**, yang lebih fleksibel untuk dashboard.

Di Colab, `share=True` memberi **link publik sementara** (berlaku ±72 jam) sehingga orang lain bisa mencoba demo tanpa instalasi. Untuk demo permanen, deploy ke Hugging Face Spaces (notebook 03).

In [ ]:
import ui_gradio

print(ui_gradio.analisis("Pengirimannya cepat, barangnya bagus!"))   # fungsi di balik UI

if not QUICK_RUN:            # tidak dijalankan saat pengujian otomatis
    ui_gradio.demo.launch(share=True)

## Ringkasan

- Deployment = menyajikan model sebagai layanan dengan **kontrak** yang jelas (input/output JSON).
- Simpan **artefak** lengkap (bobot, vocab/preprocessing, versi, metrik) dan pakai kode preprocessing yang sama saat training dan serving.
- **FastAPI**: muat model sekali, validasi input dengan Pydantic, `/health`, batch endpoint, API key dari environment variable, catat latensi.
- Uji API dengan **`TestClient`** (dan jalankan di CI).
- Ukur **p50/p95 latensi** dan **throughput**; batch dan paralelisme sangat memengaruhi throughput.
- **Gradio** untuk demo interaktif yang cepat.

## Latihan

1. **Endpoint baru.** Tambahkan endpoint `GET /info` yang mengembalikan isi `config.json` (versi, akurasi) **tanpa** API key, dan tulis tesnya di `app/tests/test_api.py`. Jalankan `pytest -q` dari folder `app/`.
2. **Ambang keyakinan.** Tambahkan field `yakin: bool` ke output (`True` jika skor tertinggi ≥ 0,7) dan parameter opsional `ambang` di request. Mengapa informasi ini berguna bagi aplikasi yang memakai API (misalnya meneruskan kasus tidak yakin ke manusia)?
3. **Rate limiting.** Batasi setiap API key maksimal 60 permintaan per menit (simpan hitungan di dict berdasarkan key dan menit), kembalikan status **429** jika terlampaui. Uji dengan loop 70 permintaan. Apa kelemahan menyimpan hitungan di memori jika server dijalankan dengan beberapa proses/replika?